# CacheStack

A `CacheStack` allows combining multiple caches into a single prioritized hierarchy.
It is a common pattern to combine a fast, local cache (like an in-memory cache) with a larger, slower, and potentially remote cache (like a database or file system).

## Automatic Hit Transfer

Whenever a hit is found in a higher-level cache, it is automatically transferred to the base cache (index 0). This ensures that frequently accessed data migrates to the fastest cache in your stack.

## Building the stack

A stack is a *list* of cache configs, fastest first — `from_config` turns the
list into a `CacheStack`, so there is no storage wiring to spell out:

In [1]:
import tempfile

from fleche import fleche, cache
from fleche.caches import BaseCache

tmp_dir = tempfile.TemporaryDirectory()

stack = BaseCache.from_config([
    # local: fast, in-process, gone when the interpreter exits
    {"template": "memory"},
    # 'remote': cloudpickle files for values, SQL for the call records
    {"template": "sql", "root": tmp_dir.name, "values": "cloudpickle"},
])
stack

CacheStack(stack=(Cache(values=ValueMemory(storage={}, remaining_depth=1), calls=CallMemory(storage={})), Cache(values=ValuePickleFile(root=PosixPath('/tmp/tmpwadquqd8/values'), secret_key=(), compress=False, remaining_depth=1), calls=Sql(url='sqlite:////tmp/tmpwadquqd8/calls.db', echo=False))))

Each level is an ordinary cache, reachable through `stack.stack` — the same two
caches you would get by building the storages by hand and passing them to
`CacheStack((local_cache, remote_cache))` yourself.

In [2]:
local_cache, remote_cache = stack.stack


@fleche
def expensive_computation(x):
    print(f"Computing {x}...")
    return x * 10


# Pre-populate the remote cache
with cache(remote_cache):
    expensive_computation(42)

print("Remote cache contains 42:", remote_cache.contains(expensive_computation.fleche.digest(42)))
print("Local cache contains 42: ", local_cache.contains(expensive_computation.fleche.digest(42)))

Computing 42...
Remote cache contains 42: True
Local cache contains 42:  False


## Automatic hit transfer in action

`local_cache` comes first in the stack, so it wins on lookup — and a hit found
further up is copied down into it.

In [3]:
with cache(stack):
    # Loading 42 will hit remote_cache and automatically transfer it to local_cache
    print("Calling expensive_computation(42) via stack...")
    result = expensive_computation(42)
    print(f"Result: {result}")

print("Local cache now contains 42:", local_cache.contains(expensive_computation.fleche.digest(42)))

# Cleanup
tmp_dir.cleanup()

Calling expensive_computation(42) via stack...
Result: 420
Local cache now contains 42: True
